                # SIMBA

                Sample trajectories, identify difficult examples, and add reflective rules or demonstrations in a sequence of improvement steps.

                **Use it when:** You want iterative, example-driven improvement and can tolerate a relatively expensive reflective search.

                **What compilation changes:** The selected program may add rules or demonstrations; this rerun's artifact shows which mechanism won.

                Important configuration in this benchmark:

                - six steps and four candidates in the full profile
- batch size capped at eight
- at most two demonstrations
- seed 42

                Every notebook loads the same 74-row dataset and frozen, pair-grouped
                train/validation/test membership before it can compile anything.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import artifact_paths, learned_program_preview, verify_prompt_artifact
from chapter06.optimizer_runtime import (
    format_result,
    load_frozen_examples,
    published_result,
    run_optimizer,
    split_summary,
)

OPTIMIZER = 'simba'
splits = load_frozen_examples()
RUN_LIVE = os.getenv("CHAPTER06_RUN_LIVE", "0") == "1"
print(f"optimizer={OPTIMIZER!r}; live={RUN_LIVE}")
print(split_summary(splits))

optimizer='simba'; live=False
train=36 (human=18, AI=18); validation=18 (human=9, AI=9); test=20 (human=10, AI=10)


                ## Compile shape

                This is the essential DSPy call used by the shared executable runner:

                ```python
                optimizer = dspy.SIMBA(
    metric=exact_match, bsize=8, num_candidates=profile.simba_candidates,
    max_steps=profile.simba_steps, max_demos=2, prompt_model=reflection_lm,
)
optimized_detector = optimizer.compile(detector, trainset=trainset, seed=42)
                ```

                `compile` returns a program. The shared runner then evaluates that program on the
                untouched 20-example test split. The baseline has its own notebook; all other
                notebooks report the optimized program's final test accuracy directly.

In [2]:
if RUN_LIVE:
    live_run = run_optimizer(
        OPTIMIZER,
        splits=splits,
    )
    result = live_run.summary()
else:
    result = published_result(OPTIMIZER)

print(format_result(result))
print()
print(artifact_paths(OPTIMIZER))

optimizer: simba
task model: openai/gpt-5.6-luna
final test accuracy: 70.0% (14/20)
optimization time: 1130.0s

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/simba.json
- canonical prompt: chapter06/results/final_prompts/simba.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Compare the final training trajectory with the locked test score. A large gap is evidence that reflective search can still overfit a small benchmark.

The saved output above uses the checked-in result so opening or running the notebook
is cheap. Set `CHAPTER06_RUN_LIVE=1` before launching Jupyter to execute the real
optimizer; prompt optimizers require an OpenAI key, while weight optimizers also
require the local PyTorch/Transformers stack. The next cell previews the published
program artifact.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (1288 characters):
Decide whether the supplied passage was generated by an AI.

When given short technical documentation or procedural instructions, do not classify it as AI merely because it uses numbered steps, concise wording, or a mechanical structure. Weigh source-like traits more heavily: exact domain API names, narrowly scoped instructions, inconsistent punctuation, awkward grammar, and lack of explanatory padding are compatible with human-authored documentation. For passages like this Spark procedure, treat the specific API references and lightly edited phrasing as evidence for `is_ai: false` unless stronger generation signals are present.

When the input is short, narrowly scoped technical or procedural documentation containing exact framework-specific identifiers—such as RDD, StructType, createDataFrame, and SparkSession—treat those source-like details as stronger evidence than numbered steps, repetitive wording, mechanical concis

## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Keep the test set untouched until the optimizer returns,
then report final accuracy as `correct / test examples` so every optimizer is easy
to compare. Use the separate baseline notebook when you also need uplift.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.